# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. All references to entities—record sets, fields, and columns—use their Croissant `@id` as required for interoperability and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and record sets from the FAIR² dataset using the `mlcroissant` API.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs, all referenced by `@id` fields.

Let's inspect the record sets in the dataset metadata (using their `@id`), and for each, display their fields and types.

In [ ]:
# List all available record sets (`@id`) in metadata
record_sets = []
if hasattr(metadata, 'record_set') and metadata.record_set:
    for rs in metadata.record_set:
        # Each rs object has @id and other metadata
        record_sets.append(rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs)

print('Available record sets by @id:')
for rsid in record_sets:
    print('  -', rsid)

# If record sets are not in the metadata (as in early JSON-LD schemas), attempt to list record set IDs from dataset
if not record_sets:
    print("\nNo 'recordSet' objects found in metadata. Attempting to enumerate available record sets via dataset...\n")
    available_sets = dataset.record_set_ids  # mlcroissant convenience property
    for rsid in available_sets:
        print('  -', rsid)
    record_sets = list(available_sets)


# List fields for each record set, using their `@id`
fields_dict = {}  # {record_set_id: [field_ids]}

for rsid in record_sets:
    try:
        fields = dataset.fields(record_set=rsid)
        field_ids = [f['@id'] for f in fields]
        fields_dict[rsid] = field_ids
        print(f"\nFields for record set @id '{rsid}':")
        for fid in field_ids:
            print('  -', fid)
    except Exception as e:
        print(f"\nNo fields accessible for record set @id '{rsid}': {e}")

## 3. Data Extraction
Load data from a specific record set using its `@id`, and examine the available columns (referenced by their `@id` as well).

In [ ]:
# Select which record set to load (by `@id`). If only one, default to that.
if len(record_sets) == 0:
    raise RuntimeError("No record sets found in the Croissant metadata.")
record_set_id = record_sets[0]
print(f"\nExtracting records for record set @id: {record_set_id}")

# Load all records for this record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

print(f"\nColumns available in '{record_set_id}': (referenced by their @id)")
print(df.columns.tolist())

# Preview data
df.head()

## 4. Exploratory Data Analysis (EDA)
Perform simple data processing using column `@id` references.

*Example: Filter on a numeric field, normalize it, and optionally group by a categorical field—all by their `@id`.*

In [ ]:
# Identify a likely numeric field from columns (@id)
from pandas.api.types import is_numeric_dtype

# Try to guess a numeric field
numeric_field_candidates = [col for col in df.columns if is_numeric_dtype(df[col])]
if not numeric_field_candidates:
    # Try to cast any column that looks numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            pass
    numeric_field_candidates = [col for col in df.columns if is_numeric_dtype(df[col])]

if not numeric_field_candidates:
    raise RuntimeError("Could not find any numeric fields for EDA. Please inspect df.columns.")

numeric_field_id = numeric_field_candidates[0]
print(f"Using '{numeric_field_id}' as the sample numeric field for analysis.")

# Define a filtering threshold (use median or a simple value for demonstration)
threshold = df[numeric_field_id].median()
filtered_df = df[df[numeric_field_id] > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (standard score)
normalized_field = f"{numeric_field_id}_normalized"
filtered_df[normalized_field] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)

print(f"\nNormalized {numeric_field_id} for filtered records (showing head):")
print(filtered_df[[numeric_field_id, normalized_field]].head())

# Try grouping by a categorical field if possible
categorical_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
group_field = categorical_fields[0] if categorical_fields else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by '{group_field}' (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field, and its relation to the group field if defined.

*All axes are labeled with column `@id`.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id], bins=25, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.show()

if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x=group_field, y=numeric_field_id)
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded metadata and record sets from the FAIR² dataset using the `mlcroissant` library. All data entities were referenced by their `@id` fields for reproducibility. We explored available record sets and fields, performed basic filtering and normalization on a numeric field, and visualized key distributions. This workflow enables transparent, standards-based exploration and analysis of Croissant-described datasets.